# 04 — Weekly Team Strength

This notebook updates team strength sequentially using the new in season evidence created in `03_Inseason_Feature_Engineering.ipynb`.

The original preseason model remains unchanged.

The weekly model follows a sequential principle:

> The previous week's team strength is the prior. The most recently completed week provides new evidence.

For each target week, this notebook:

- Loads the frozen preseason ratings for reference
- Loads the previous week's team strength state
- Loads the target week's new in season feature table
- Applies the opponent adjusted weekly strength update
- Preserves separate offensive and defensive weekly signals
- Calculates current weekly rankings
- Compares rank and strength movement to the previous week
- Saves a separate weekly team strength file

No original preseason file or prior weekly file is overwritten.

In [29]:
from pathlib import Path

import numpy as np
import pandas as pd


## Paths and Weekly Settings

This notebook expects the output from:

`03_Inseason_Feature_Engineering.ipynb`

For Week 3:

- `TARGET_WEEK = 3`
- `PRIOR_WEEK = 2`

The Week 2 team strength file becomes the entering prior for Week 3.

In [30]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

SEASON = 2026
TARGET_WEEK = 12
PRIOR_WEEK = TARGET_WEEK - 1

PRESEASON_STRENGTH_PATH = (
    PROCESSED_DIR
    / "2026_team_strength.parquet"
)

PRIOR_WEEK_STRENGTH_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{PRIOR_WEEK:02d}_sequential_team_strength.parquet"
)

INSEASON_FEATURES_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_inseason_features.parquet"
)

OUTPUT_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_sequential_team_strength.parquet"
)

print("Preseason ratings:", PRESEASON_STRENGTH_PATH)
print("Prior-week ratings:", PRIOR_WEEK_STRENGTH_PATH)
print("New in-season features:", INSEASON_FEATURES_PATH)
print("Weekly output:", OUTPUT_PATH)

Preseason ratings: ..\..\data\processed\2026_team_strength.parquet
Prior-week ratings: ..\..\data\processed\weekly\week_11_sequential_team_strength.parquet
New in-season features: ..\..\data\processed\weekly\week_12_inseason_features.parquet
Weekly output: ..\..\data\processed\weekly\week_12_sequential_team_strength.parquet


# Load Frozen Preseason Team Strength

The preseason team strength file is the final output of the original `05_Team_Strength_Model.ipynb`.

Its `team_strength` value is already expressed in point differential terms, so it can serve directly as the prior for weekly updates.

We immediately rename the important preseason columns so it is impossible to confuse the original ratings with the new weekly ratings.


In [31]:
preseason = pd.read_parquet(
    PRESEASON_STRENGTH_PATH
).copy()

required_preseason_columns = [
    "team",
    "team_strength",
    "baseline_team_strength",
    "personnel_adjustment",
    "personnel_strength",
    "roster_continuity",
    "roster_continuity_adjustment"
]

missing_preseason = [
    column
    for column in required_preseason_columns
    if column not in preseason.columns
]

if missing_preseason:
    raise KeyError(
        "Missing required preseason columns: "
        + ", ".join(missing_preseason)
    )

preseason = preseason.rename(
    columns={
        "strength_rank": "preseason_strength_rank",
        "team_strength": "preseason_team_strength",
        "baseline_team_strength": "preseason_baseline_team_strength",
        "personnel_adjustment": "preseason_personnel_adjustment",
        "personnel_strength": "preseason_personnel_strength",
        "roster_continuity": "preseason_roster_continuity",
        "roster_continuity_adjustment": "preseason_roster_continuity_adjustment"
    }
)

print("Preseason teams:", len(preseason))
print("Unique teams:", preseason["team"].nunique())

display(
    preseason[
        [
            "preseason_strength_rank",
            "team",
            "preseason_team_strength",
            "preseason_baseline_team_strength",
            "preseason_personnel_adjustment",
            "preseason_roster_continuity_adjustment"
        ]
    ].sort_values(
        "preseason_strength_rank"
    )
)


Preseason teams: 32
Unique teams: 32


,preseason_strength_rank,team,preseason_team_strength,preseason_baseline_team_strength,preseason_personnel_adjustment,preseason_roster_continuity_adjustment
0,1,BUF,4.816514,4.125764,0.934506,0.260091
1,2,LA,4.762452,3.257133,2.100811,0.489665
2,3,SEA,4.363421,3.515976,0.635683,0.932073
3,4,DET,4.018565,3.374644,1.041923,0.037533
4,5,DEN,3.569507,2.355154,0.979753,1.253003
5,6,BAL,3.074111,2.710864,0.636254,-0.037012
6,7,HOU,2.810133,2.193179,1.173833,-0.174691
7,8,GB,2.536317,1.718369,0.882628,0.576742
8,9,PHI,2.451919,2.324982,0.264690,-0.063756
9,10,SF,2.366788,1.805709,0.692939,0.290631


# Load Prior-Week Team Strength

The previous week's team strength file is the starting point for the new weekly update.

For Week 3, the model begins with the Week 2 ratings.

This prevents older games from being counted repeatedly and creates a true sequential rating system.

In [32]:
prior_week = pd.read_parquet(
    PRIOR_WEEK_STRENGTH_PATH
).copy()

required_prior_columns = [
    "team",
    "weekly_strength_rank",
    "weekly_team_strength",
    "cumulative_offense_adjustment",
    "cumulative_defense_adjustment"
]

missing_prior = [
    column
    for column in required_prior_columns
    if column not in prior_week.columns
]

if missing_prior:
    raise KeyError(
        "Missing required prior-week columns: "
        + ", ".join(missing_prior)
    )

prior_week = prior_week[
    required_prior_columns
].rename(
    columns={
        "weekly_strength_rank":
            "prior_week_strength_rank",
        "weekly_team_strength":
            "prior_week_team_strength",
        "cumulative_offense_adjustment":
            "prior_cumulative_offense_adjustment",
        "cumulative_defense_adjustment":
            "prior_cumulative_defense_adjustment"
    }
)

print(
    f"Week {PRIOR_WEEK} teams:",
    len(prior_week)
)

print(
    "Unique teams:",
    prior_week["team"].nunique()
)

display(
    prior_week.sort_values(
        "prior_week_strength_rank"
    )
)

Week 11 teams: 32
Unique teams: 32


,team,prior_week_strength_rank,prior_week_team_strength,prior_cumulative_offense_adjustment,prior_cumulative_defense_adjustment
0,BUF,1,8.151564,11.999972,1.546936
1,JAX,2,6.485660,8.950045,9.447010
2,LA,3,5.981859,2.223856,6.192249
3,SF,4,5.208822,10.913442,1.585406
4,KC,5,5.070234,4.346574,7.354966
5,DET,6,4.974895,6.925970,1.208649
6,SEA,7,4.310075,-1.127478,8.390915
7,CIN,8,4.292347,10.288631,2.321309
8,DEN,9,3.953856,-0.891564,3.473258
9,CHI,10,3.472897,5.291976,3.506798


# Load New Weekly In-Season Features

These features contain the new evidence generated in notebook `03`.

For Week 3, they describe each team's Week 2 performance relative to its entering Week 2 expectation.

The primary values used here are:

- `weekly_strength_update`
- `weekly_offense_adjustment`
- `weekly_defense_adjustment`

The overall strength update is opponent adjusted and uses diminishing returns for extreme single game results.

In [33]:
inseason = pd.read_parquet(
    INSEASON_FEATURES_PATH
).copy()

required_inseason_columns = [
    "team",
    "games_in_update",
    "entering_team_strength",
    "entering_strength_rank",
    "average_opponent_strength",
    "average_opponent_rank",
    "actual_margin",
    "expected_margin",
    "performance_residual",
    "weekly_strength_update",
    "weekly_offense_adjustment",
    "weekly_defense_adjustment"
]

missing_inseason = [
    column
    for column in required_inseason_columns
    if column not in inseason.columns
]

if missing_inseason:
    raise KeyError(
        "Missing required in-season columns: "
        + ", ".join(missing_inseason)
    )

print("In-season teams:", len(inseason))
print(
    "Unique teams:",
    inseason["team"].nunique()
)

display(
    inseason[
        required_inseason_columns
    ].sort_values(
        "weekly_strength_update",
        ascending=False
    )
)

In-season teams: 26
Unique teams: 26


,team,games_in_update,entering_team_strength,entering_strength_rank,average_opponent_strength,average_opponent_rank,actual_margin,expected_margin,performance_residual,weekly_strength_update,weekly_offense_adjustment,weekly_defense_adjustment
22,SF,1,5.208822,4,3.069441,11.0,25.0,3.639381,21.360619,1.364544,1.811258,2.168950
9,HOU,1,1.556349,13,-3.273874,22.0,23.0,6.330223,16.669777,1.246193,0.223384,2.981076
12,KC,1,5.070234,5,-6.133545,30.0,27.0,12.703779,14.296221,1.155507,1.316146,2.273838
6,DAL,1,-0.672234,17,-7.194555,32.0,21.0,8.022321,12.977679,1.093772,3.826358,-1.215950
18,NYG,1,-2.056090,19,6.485660,2.0,5.0,-7.041750,12.041750,1.044483,0.139366,1.097058
19,NYJ,1,-6.983012,31,-4.000654,25.0,3.0,-4.482358,7.482358,0.733168,0.346129,-0.196178
20,PHI,1,-4.148071,27,1.547719,14.0,3.0,-4.195790,7.195790,0.709566,-0.645807,1.211886
25,WAS,1,-1.373597,18,4.292347,8.0,2.0,-4.165944,6.165944,0.620994,-0.092883,0.714809
8,DET,1,4.974895,6,1.202707,15.0,10.0,5.272188,4.727812,0.488135,1.141255,0.448948
7,DEN,1,3.953856,9,-4.433542,28.0,14.0,9.887398,4.112602,0.428384,1.979896,-0.212412


# Merge Prior Week State with New Evidence

The weekly model now combines three pieces of information:

1. Frozen preseason information for long term reference
2. Previous week's team strength as the current prior
3. Most recent week's opponent adjusted performance as new evidence

All 32 teams must remain in the table.

In [34]:
weekly_strength = (
    preseason
    .merge(
        prior_week,
        on="team",
        how="left",
        validate="one_to_one"
    )
    .merge(
        inseason,
        on="team",
        how="left",
        validate="one_to_one"
    )
)

weekly_fill_columns = [
    "games_in_update",
    "weekly_strength_update",
    "weekly_offense_adjustment",
    "weekly_defense_adjustment"
]

weekly_strength[
    weekly_fill_columns
] = (
    weekly_strength[
        weekly_fill_columns
    ]
    .fillna(0.0)
)

print("Merged teams:", len(weekly_strength))
print(
    "Unique teams:",
    weekly_strength["team"].nunique()
)

print(
    "Missing prior-week strengths:",
    weekly_strength[
        "prior_week_team_strength"
    ].isna().sum()
)

Merged teams: 32
Unique teams: 32
Missing prior-week strengths: 0


# Sequential Weekly Rating Update

The previous week's team strength is the starting point.

The current week's opponent adjusted performance update is then applied:

`weekly_team_strength = prior_week_team_strength + weekly_strength_update`

Therefore, for Week 3:

`Week 3 Strength = Week 2 Strength + Week 2 Performance Update`

Older results remain embedded in the prior week rating and are not counted again.

The offensive and defensive weekly adjustments remain available separately for the weekly scoring model.

In [35]:
weekly_strength["team_strength_change"] = (
    weekly_strength["weekly_strength_update"]
)

weekly_strength["weekly_team_strength"] = (
    weekly_strength["prior_week_team_strength"]
    + weekly_strength["team_strength_change"]
)

display(
    weekly_strength[
        [
            "team",
            "prior_week_team_strength",
            "weekly_offense_adjustment",
            "weekly_defense_adjustment",
            "team_strength_change",
            "weekly_team_strength"
        ]
    ].sort_values(
        "weekly_team_strength",
        ascending=False
    )
)

,team,prior_week_team_strength,weekly_offense_adjustment,weekly_defense_adjustment,team_strength_change,weekly_team_strength
0,BUF,8.151564,0.164618,2.022310,0.265821,8.417385
9,SF,5.208822,1.811258,2.168950,1.364544,6.573366
12,KC,5.070234,1.316146,2.273838,1.155507,6.225741
1,LA,5.981859,0.000000,0.000000,0.000000,5.981859
3,DET,4.974895,1.141255,0.448948,0.488135,5.463030
11,JAX,6.485660,-0.930950,0.026743,-1.044483,5.441177
4,DEN,3.953856,1.979896,-0.212412,0.428384,4.382240
2,SEA,4.310075,0.000000,0.000000,0.000000,4.310075
20,CIN,4.292347,-0.605356,0.202336,-0.620994,3.671352
18,CHI,3.472897,-0.528775,1.478917,-0.032602,3.440295


In [36]:
weekly_strength["cumulative_offense_adjustment"] = (
    weekly_strength["prior_cumulative_offense_adjustment"]
    + weekly_strength["weekly_offense_adjustment"]
)

weekly_strength["cumulative_defense_adjustment"] = (
    weekly_strength["prior_cumulative_defense_adjustment"]
    + weekly_strength["weekly_defense_adjustment"]
)

assert weekly_strength[
    "cumulative_offense_adjustment"
].notna().all()

assert weekly_strength[
    "cumulative_defense_adjustment"
].notna().all()

# Updated Weekly Rankings

The current weekly rank is based on `weekly_team_strength`.

Ranking movement is now measured relative to the immediately previous week rather than the preseason ranking.

A positive `rank_change` means the team moved up.

For example:

- Week 2 rank = 8
- Week 3 rank = 5
- Rank change = +3

In [37]:
weekly_strength = weekly_strength.sort_values(
    [
        "weekly_team_strength",
        "prior_week_team_strength"
    ],
    ascending=[False, False]
).reset_index(drop=True)

weekly_strength["weekly_strength_rank"] = (
    np.arange(
        1,
        len(weekly_strength) + 1
    )
)

weekly_strength["rank_change"] = (
    weekly_strength["prior_week_strength_rank"]
    - weekly_strength["weekly_strength_rank"]
)

weekly_comparison_table = (
    weekly_strength[
        [
            "weekly_strength_rank",
            "team",
            "weekly_team_strength",
            "prior_week_strength_rank",
            "prior_week_team_strength",
            "team_strength_change",
            "rank_change"
        ]
    ]
    .copy()
)

display(weekly_comparison_table)

,weekly_strength_rank,team,weekly_team_strength,prior_week_strength_rank,prior_week_team_strength,team_strength_change,rank_change
0,1,BUF,8.417385,1,8.151564,0.265821,0
1,2,SF,6.573366,4,5.208822,1.364544,2
2,3,KC,6.225741,5,5.070234,1.155507,2
3,4,LA,5.981859,3,5.981859,0.000000,-1
4,5,DET,5.463030,6,4.974895,0.488135,1
5,6,JAX,5.441177,2,6.485660,-1.044483,-4
6,7,DEN,4.382240,9,3.953856,0.428384,2
7,8,SEA,4.310075,7,4.310075,0.000000,-1
8,9,CIN,3.671352,8,4.292347,-0.620994,-1
9,10,CHI,3.440295,10,3.472897,-0.032602,0


# Biggest Weekly Movers

This table is diagnostic only.

It shows which teams experienced the largest overall rating changes from the previous week to the current week.

The underlying projection model still retains the full preseason, offense, defense, opponent and weekly information.

In [38]:
mover_columns = [
    "team",
    "prior_week_team_strength",
    "team_strength_change",
    "weekly_team_strength",
    "prior_week_strength_rank",
    "weekly_strength_rank",
    "rank_change"
]

biggest_risers = (
    weekly_strength[
        mover_columns
    ]
    .sort_values(
        "team_strength_change",
        ascending=False
    )
    .head(10)
)

biggest_fallers = (
    weekly_strength[
        mover_columns
    ]
    .sort_values(
        "team_strength_change",
        ascending=True
    )
    .head(10)
)

print("BIGGEST RISERS")
display(biggest_risers)

print()
print("BIGGEST FALLERS")
display(biggest_fallers)

BIGGEST RISERS


,team,prior_week_team_strength,team_strength_change,weekly_team_strength,prior_week_strength_rank,weekly_strength_rank,rank_change
1,SF,5.208822,1.364544,6.573366,4,2,2
10,HOU,1.556349,1.246193,2.802542,13,11,2
2,KC,5.070234,1.155507,6.225741,5,3,2
15,DAL,-0.672234,1.093772,0.421538,17,16,1
18,NYG,-2.056090,1.044483,-1.011607,19,19,0
29,NYJ,-6.983012,0.733168,-6.249844,31,30,1
23,PHI,-4.148071,0.709566,-3.438505,27,24,3
17,WAS,-1.373597,0.620994,-0.752603,18,18,0
4,DET,4.974895,0.488135,5.463030,6,5,1
6,DEN,3.953856,0.428384,4.382240,9,7,2



BIGGEST FALLERS


,team,prior_week_team_strength,team_strength_change,weekly_team_strength,prior_week_strength_rank,weekly_strength_rank,rank_change
12,MIN,3.069441,-1.364544,1.704897,11,13,-2
25,IND,-3.273874,-1.246193,-4.520067,22,26,-4
30,ARI,-6.133545,-1.155507,-7.289052,30,31,-1
31,TEN,-7.194555,-1.093772,-8.288327,32,32,0
5,JAX,6.485660,-1.044483,5.441177,2,6,-4
26,LAC,-4.000654,-0.733168,-4.733822,25,27,-2
13,PIT,1.547719,-0.709566,0.838153,14,14,0
8,CIN,4.292347,-0.620994,3.671352,8,9,-1
14,TB,1.202707,-0.488135,0.714572,15,15,0
27,LV,-4.433542,-0.428384,-4.861926,28,28,0


# Update Diagnostics

These diagnostics check the size and distribution of the sequential weekly update.

The comparison is now:

`Prior Week Strength → Current Week Strength`

rather than:

`Preseason Strength → Current Week Strength`

In [39]:
movement = (
    weekly_strength[
        "team_strength_change"
    ].abs()
)

print(
    f"WEEK {PRIOR_WEEK} → "
    f"WEEK {TARGET_WEEK} TEAM-STRENGTH MOVEMENT"
)

print(
    weekly_strength[
        "team_strength_change"
    ]
    .describe()
    .round(3)
)

print()

print(
    "Mean absolute movement:",
    round(
        movement.mean(),
        3
    )
)

print(
    "Maximum absolute movement:",
    round(
        movement.max(),
        3
    )
)

print(
    "Team with largest movement:",
    weekly_strength.loc[
        movement.idxmax(),
        "team"
    ]
)

print()
print("RATING DISTRIBUTIONS")

distribution_check = pd.DataFrame(
    {
        "metric": [
            f"Week {PRIOR_WEEK} Team Strength",
            f"Week {TARGET_WEEK} Team Strength"
        ],
        "mean": [
            weekly_strength[
                "prior_week_team_strength"
            ].mean(),
            weekly_strength[
                "weekly_team_strength"
            ].mean()
        ],
        "std": [
            weekly_strength[
                "prior_week_team_strength"
            ].std(),
            weekly_strength[
                "weekly_team_strength"
            ].std()
        ],
        "min": [
            weekly_strength[
                "prior_week_team_strength"
            ].min(),
            weekly_strength[
                "weekly_team_strength"
            ].min()
        ],
        "max": [
            weekly_strength[
                "prior_week_team_strength"
            ].max(),
            weekly_strength[
                "weekly_team_strength"
            ].max()
        ]
    }
)

display(
    distribution_check.round(3)
)

WEEK 11 → WEEK 12 TEAM-STRENGTH MOVEMENT
count    32.000
mean     -0.000
std       0.761
min      -1.365
25%      -0.521
50%       0.000
75%       0.521
max       1.365
Name: team_strength_change, dtype: float64

Mean absolute movement: 0.582
Maximum absolute movement: 1.365
Team with largest movement: SF

RATING DISTRIBUTIONS


,metric,mean,std,min,max
0,Week 11 Team Strength,0.002,4.373,-7.195,8.152
1,Week 12 Team Strength,0.002,4.574,-8.288,8.417


# Sanity Checks

These checks verify the sequential weekly update.

The weekly file should:

- contain exactly 32 unique teams
- preserve the frozen preseason information
- contain a valid prior week rating for every team
- contain no missing current weekly strength values
- have `team_strength_change` equal to the new opponent adjusted weekly update
- calculate current strength as prior week strength plus the weekly update
- never overwrite the preseason or prior week files

In [40]:
assert len(weekly_strength) == 32, (
    "Expected 32 teams in weekly strength table."
)

assert (
    weekly_strength["team"].nunique()
    == 32
), (
    "Expected 32 unique teams."
)

assert weekly_strength[
    "preseason_team_strength"
].notna().all(), (
    "Missing frozen preseason ratings."
)

assert weekly_strength[
    "prior_week_team_strength"
].notna().all(), (
    "Missing prior-week ratings."
)

assert weekly_strength[
    "weekly_team_strength"
].notna().all(), (
    "Missing current weekly ratings."
)

assert np.allclose(
    weekly_strength[
        "team_strength_change"
    ],
    weekly_strength[
        "weekly_strength_update"
    ]
), (
    "Team-strength change does not match "
    "the opponent-adjusted weekly update."
)

assert np.allclose(
    weekly_strength[
        "weekly_team_strength"
    ],
    (
        weekly_strength[
            "prior_week_team_strength"
        ]
        + weekly_strength[
            "team_strength_change"
        ]
    )
), (
    "Sequential weekly strength formula "
    "is inconsistent."
)

played_mask = weekly_strength["games_in_update"] > 0

assert np.allclose(
    weekly_strength.loc[
        played_mask, "entering_team_strength"
    ],
    weekly_strength.loc[
        played_mask, "prior_week_team_strength"
    ]
), (
    "Notebook 03 entering strength does not match "
    "the prior-week team strength."
)

bye_mask = ~played_mask

assert np.allclose(
    weekly_strength.loc[
        bye_mask, "weekly_team_strength"
    ],
    weekly_strength.loc[
        bye_mask, "prior_week_team_strength"
    ]
), "Bye-week teams did not retain their previous strength."

print(
    "All sequential weekly team-strength "
    "sanity checks passed."
)

All sequential weekly team-strength sanity checks passed.


# Final Weekly Team Strength Table

The saved file contains the full state needed by the weekly projection pipeline.

This includes:

- current weekly strength and rank
- previous week's strength and rank
- week to week movement
- frozen preseason information
- opponent adjusted performance information
- separate offensive and defensive weekly signals

The compact ranking table shown earlier is only for interpretation. The saved dataset retains the additional model information.

In [41]:
final_columns = [
    "weekly_strength_rank",
    "team",
    "weekly_team_strength",

    "prior_week_strength_rank",
    "prior_week_team_strength",
    "team_strength_change",
    "rank_change",

    "games_in_update",
    "average_opponent_rank",
    "average_opponent_strength",
    "actual_margin",
    "expected_margin",
    "performance_residual",
    "weekly_strength_update",

    "weekly_offense_adjustment",
    "weekly_defense_adjustment",
    "cumulative_offense_adjustment",
    "cumulative_defense_adjustment",

    "preseason_strength_rank",
    "preseason_team_strength",
    "preseason_baseline_team_strength",
    "preseason_personnel_adjustment",
    "preseason_personnel_strength",
    "preseason_roster_continuity",
    "preseason_roster_continuity_adjustment"
]

weekly_team_strength = (
    weekly_strength[
        final_columns
    ]
    .copy()
)

display(
    weekly_team_strength
)

,weekly_strength_rank,team,weekly_team_strength,prior_week_strength_rank,prior_week_team_strength,team_strength_change,rank_change,games_in_update,average_opponent_rank,average_opponent_strength,...,weekly_defense_adjustment,cumulative_offense_adjustment,cumulative_defense_adjustment,preseason_strength_rank,preseason_team_strength,preseason_baseline_team_strength,preseason_personnel_adjustment,preseason_personnel_strength,preseason_roster_continuity,preseason_roster_continuity_adjustment
0,1,BUF,8.417385,1,8.151564,0.265821,0,1.0,29.0,-4.840968,...,2.022310,12.164589,3.569246,1,4.816514,4.125764,0.934506,0.623004,0.573034,0.260091
1,2,SF,6.573366,4,5.208822,1.364544,2,1.0,11.0,3.069441,...,2.168950,12.724700,3.754356,10,2.366788,1.805709,0.692939,0.461959,0.577778,0.290631
2,3,KC,6.225741,5,5.070234,1.155507,2,1.0,30.0,-6.133545,...,2.273838,5.662719,9.628805,13,1.302244,1.484935,-0.187743,-0.125162,0.510870,-0.140090
3,4,LA,5.981859,3,5.981859,0.000000,-1,0.0,NaN,NaN,...,0.000000,2.223856,6.192249,2,4.762452,3.257133,2.100811,1.400540,0.608696,0.489665
4,5,DET,5.463030,6,4.974895,0.488135,1,1.0,15.0,1.202707,...,0.448948,8.067225,1.657596,4,4.018565,3.374644,1.041923,0.694616,0.538462,0.037533
5,6,JAX,5.441177,2,6.485660,-1.044483,-4,1.0,19.0,-2.056090,...,0.026743,8.019096,9.473752,12,1.713136,1.746953,-0.223729,-0.149153,0.563830,0.200841
6,7,DEN,4.382240,9,3.953856,0.428384,2,1.0,28.0,-4.433542,...,-0.212412,1.088332,3.260846,5,3.569507,2.355154,0.979753,0.653169,0.727273,1.253003
7,8,SEA,4.310075,7,4.310075,0.000000,-1,0.0,NaN,NaN,...,0.000000,-1.127478,8.390915,3,4.363421,3.515976,0.635683,0.423788,0.677419,0.932073
8,9,CIN,3.671352,8,4.292347,-0.620994,-1,1.0,18.0,-1.373597,...,0.202336,9.683274,2.523645,21,-0.739434,-1.268642,0.286293,0.190862,0.643678,0.714864
9,10,CHI,3.440295,10,3.472897,-0.032602,0,1.0,23.0,-3.331436,...,1.478917,4.763201,4.985715,19,-0.051934,-0.069709,0.002783,0.001856,0.537634,0.032209


# Save Weekly Team Strength

This writes only to:

`data/processed/weekly/`

The original:

`data/processed/2026_team_strength.parquet`

remains untouched.


In [42]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

weekly_team_strength.to_parquet(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)
print()
print(
    "Original preseason file remains:",
    PRESEASON_STRENGTH_PATH
)


Saved: ..\..\data\processed\weekly\week_12_sequential_team_strength.parquet

Original preseason file remains: ..\..\data\processed\2026_team_strength.parquet


# What This Means for Week 3

At the end of this notebook, every team has:

- its frozen preseason rating
- its Week 2 entering rating and rank
- its opponent adjusted Week 2 performance
- its Week 2 offensive and defensive signals
- its Week 2 → Week 3 rating movement
- its updated Week 3 team strength rating
- its updated Week 3 rank

The next stage will apply the Week 3 injury update and then use the resulting team state to generate Week 3 game projections.

The weekly ranking table is diagnostic only.

The saved weekly team strength file retains the additional information required by the projection pipeline.